# Secure Inference: HE vs MPC

Two approaches to running model inference on sensitive data **without revealing the input**.

| | HE (Homomorphic Encryption) | MPC (Multi-Party Computation) |
|---|---|---|
| **Library** | [TenSEAL](https://github.com/OpenMined/TenSEAL) (OpenMined) | [CrypTen](https://github.com/facebookresearch/CrypTen) (Meta) |
| **How it works** | Compute directly on ciphertext | Split data into secret shares across parties |
| **Parties needed** | 1 (single server) | 2+ (multi-party) |
| **ReLU / non-linear** | Polynomial approximation only | Full support |
| **Best for** | Small models (MLP, logistic regression) | Large models (DenseNet, BERT) |
| **Install** | `pip install tenseal` | `pip install crypten` |

This tutorial trains **one fraud detection model** and runs inference via **both** methods, comparing results.

---

## Contents

1. [Train a Plaintext Model](#1-train)
2. [HE: Encrypted Inference with TenSEAL](#2-he)
3. [MPC: Secret-Shared Inference](#3-mpc)
4. [Side-by-Side Comparison](#4-comparison)
5. [When to Use Which](#5-decision)

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

import torch
import torch.nn as nn
import numpy as np
import time

print(f'Working directory: {os.getcwd()}')

---
## 1. Train a Plaintext Model <a id='1-train'></a>

Train a fraud detection MLP on real data. This is the model both HE and MPC will use for encrypted inference.

In [ ]:
# Load real fraud data
data = np.load('data/samples/fraud/data.npz')
X_train = torch.from_numpy(data['X'][:2000])
y_train = torch.from_numpy(data['y'][:2000])
X_test = torch.from_numpy(data['X'][2000:2020])  # 20 test patients
y_test = torch.from_numpy(data['y'][2000:2020])

# Small MLP (HE can only handle small models)
model = nn.Sequential(
    nn.Linear(30, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid(),
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(30):
    pred = model(X_train).squeeze()
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()
    acc = ((preds > 0.5).float() == y_test).float().mean()
    print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
    print(f'Plaintext accuracy: {acc:.3f} on {len(X_test)} test samples')
    print(f'Plaintext predictions (first 5): {[f"{p:.4f}" for p in preds[:5].tolist()]}')

---
## 2. HE: Encrypted Inference with TenSEAL <a id='2-he'></a>

With Homomorphic Encryption, the hospital encrypts patient data, sends ciphertext to the cloud, and the cloud runs the model **on encrypted data** without ever seeing the plaintext.

```
Hospital                          Cloud
   |                                |
   |  encrypt(patient)  ────────>   |
   |                         model(ciphertext)
   |  <──────── encrypt(prediction) |
   |                                |
   decrypt(prediction)              (never saw raw data)
```

### Limitation: HE only supports polynomial operations

CKKS supports addition and multiplication, but NOT:
- Comparison (`>`, `<`)
- ReLU (requires comparison with 0)
- Sigmoid, softmax

We approximate these with **polynomial functions** — works for small models but loses accuracy on deep networks.

In [ ]:
import tenseal as ts
from fl_pets.he import create_context, encrypt, decrypt

print(f'TenSEAL {ts.__version__} (Microsoft SEAL, CKKS scheme)')

# Create encryption context
ctx = create_context(scheme='ckks', poly_mod_degree=8192)

# Extract model weights
w1, b1 = model[0].weight.detach().numpy(), model[0].bias.detach().numpy()
w2, b2 = model[2].weight.detach().numpy(), model[2].bias.detach().numpy()

def he_sigmoid_approx(x_list):
    """Approximate sigmoid: 0.5 + 0.197*x - 0.004*x^3 (degree-3 polynomial)"""
    return [max(0, min(1, 0.5 + 0.197 * x - 0.004 * x**3)) for x in x_list]

def he_linear(ctx, enc_input, weight, bias):
    """Encrypted matrix-vector multiply: y = Wx + b."""
    outputs = []
    for j in range(weight.shape[0]):
        # Dot product on encrypted data, then decrypt to get one neuron output
        enc_dot = enc_input.dot(weight[j].tolist())
        val = enc_dot.decrypt()[0] + float(bias[j])
        outputs.append(val)
    return outputs

# Run HE inference on each test patient
he_predictions = []
t0 = time.time()

for idx in range(len(X_test)):
    patient = X_test[idx].numpy().tolist()
    
    # Hospital encrypts patient data
    enc_patient = ts.ckks_vector(ctx, patient)
    
    # Cloud: Layer 1 (linear + ReLU)
    layer1_out = he_linear(ctx, enc_patient, w1, b1)
    layer1_relu = [max(0, x) for x in layer1_out]  # ReLU on decrypted (limitation of HE)
    
    # Cloud: Layer 2 (linear + sigmoid)
    enc_hidden = ts.ckks_vector(ctx, layer1_relu)
    layer2_out = he_linear(ctx, enc_hidden, w2, b2)
    sigmoid_out = he_sigmoid_approx(layer2_out)
    he_predictions.append(sigmoid_out[0])

he_time = time.time() - t0
print(f'HE inference: {len(X_test)} patients in {he_time:.2f}s ({he_time/len(X_test)*1000:.0f}ms per patient)')
print(f'HE predictions (first 5): {[f"{p:.4f}" for p in he_predictions[:5]]}')

---
## 3. MPC: Secret-Shared Inference <a id='3-mpc'></a>

With MPC, both the patient data and model weights are split into **secret shares** distributed across parties. Each party computes on its share — no single party ever sees the full data or model.

```
Party A (hospital)                  Party B (cloud)
   |                                   |
   share_A(patient)                    share_B(patient)
   share_A(weights)                    share_B(weights)
   |                                   |
   compute_A(layer1)  <--- comm --->   compute_B(layer1)   ← ReLU requires communication
   compute_A(layer2)  <--- comm --->   compute_B(layer2)
   |                                   |
   share_A(pred) ──────> reconstruct(pred)
```

### Advantage over HE: exact ReLU and sigmoid

MPC can compute non-linear operations exactly (using garbled circuits or comparison protocols), so there is **no approximation error**.

In [ ]:
# Secret sharing primitives
def secret_share(tensor, num_parties=2):
    shares = [torch.randn_like(tensor) for _ in range(num_parties - 1)]
    shares.append(tensor - sum(shares))
    return shares

def reconstruct(shares):
    return sum(shares)

# MPC layer operations (simulated on single machine)
def mpc_linear(x_shares, weight, bias):
    y_shares = []
    for i, share in enumerate(x_shares):
        y = share @ weight.T
        if i == 0:
            y = y + bias
        y_shares.append(y)
    return y_shares

def mpc_relu(x_shares):
    # In real MPC: garbled circuits. Here: reconstruct, apply, re-share.
    return secret_share(torch.relu(reconstruct(x_shares)))

def mpc_sigmoid(x_shares):
    return secret_share(torch.sigmoid(reconstruct(x_shares)))

# Run MPC inference
mpc_predictions = []
t0 = time.time()

with torch.no_grad():
    w1t, b1t = model[0].weight, model[0].bias
    w2t, b2t = model[2].weight, model[2].bias
    
    for idx in range(len(X_test)):
        patient = X_test[idx]
        
        # Secret-share the patient data
        x_shares = secret_share(patient)
        
        # Layer 1: Linear + ReLU
        x_shares = mpc_linear(x_shares, w1t, b1t)
        x_shares = mpc_relu(x_shares)
        
        # Layer 2: Linear + Sigmoid
        x_shares = mpc_linear(x_shares, w2t, b2t)
        x_shares = mpc_sigmoid(x_shares)
        
        # Reconstruct prediction
        pred = reconstruct(x_shares).item()
        mpc_predictions.append(pred)

mpc_time = time.time() - t0
print(f'MPC inference: {len(X_test)} patients in {mpc_time:.2f}s ({mpc_time/len(X_test)*1000:.0f}ms per patient)')
print(f'MPC predictions (first 5): {[f"{p:.4f}" for p in mpc_predictions[:5]]}')

---
## 4. Side-by-Side Comparison <a id='4-comparison'></a>

In [ ]:
# Compare all three methods
with torch.no_grad():
    plain_preds = model(X_test).squeeze().tolist()

print(f'{"Patient":>8} {"True":>6} {"Plain":>8} {"HE":>8} {"MPC":>8} {"HE err":>8} {"MPC err":>8}')
print('-' * 62)

he_errors = []
mpc_errors = []

for i in range(len(X_test)):
    true = int(y_test[i].item())
    plain = plain_preds[i]
    he = he_predictions[i]
    mpc = mpc_predictions[i]
    he_err = abs(plain - he)
    mpc_err = abs(plain - mpc)
    he_errors.append(he_err)
    mpc_errors.append(mpc_err)
    print(f'{i:>8} {true:>6} {plain:>8.4f} {he:>8.4f} {mpc:>8.4f} {he_err:>8.2e} {mpc_err:>8.2e}')

print()
print(f'Mean absolute error vs plaintext:')
print(f'  HE:  {np.mean(he_errors):.6f}  (polynomial approximation introduces error)')
print(f'  MPC: {np.mean(mpc_errors):.8f}  (exact computation, only floating-point rounding)')
print()
print(f'Inference time ({len(X_test)} patients):')
print(f'  HE:  {he_time:.2f}s  ({he_time/len(X_test)*1000:.0f}ms/patient)')
print(f'  MPC: {mpc_time:.2f}s  ({mpc_time/len(X_test)*1000:.0f}ms/patient)')
print()

# Accuracy comparison
plain_acc = ((torch.tensor(plain_preds) > 0.5).float() == y_test).float().mean()
he_acc = ((torch.tensor(he_predictions) > 0.5).float() == y_test).float().mean()
mpc_acc = ((torch.tensor(mpc_predictions) > 0.5).float() == y_test).float().mean()
print(f'Classification accuracy:')
print(f'  Plaintext: {plain_acc:.3f}')
print(f'  HE:        {he_acc:.3f}')
print(f'  MPC:       {mpc_acc:.3f}')

---
## 5. When to Use Which <a id='5-decision'></a>

| Criteria | Choose HE | Choose MPC |
|----------|-----------|------------|
| **Model size** | Small (<10K params) | Any size (DenseNet, BERT) |
| **Non-linear layers** | None or few (polynomial approx) | Unlimited (exact ReLU, sigmoid) |
| **Parties** | Single server (hospital → cloud) | 2+ parties required |
| **Communication** | One round-trip (send ciphertext, get result) | Multi-round (per non-linear layer) |
| **Accuracy** | Approximate (polynomial error) | Exact (matches plaintext) |
| **Use case** | Simple scoring (logistic regression, linear) | Full neural network inference |

### Production libraries

| Library | Type | Maintainer | Validated on |
|---------|------|-----------|-------------|
| **TenSEAL** | HE (CKKS/BFV) | OpenMined | Small MLPs, linear models |
| **Microsoft SEAL** | HE | Microsoft | Underlying engine for TenSEAL |
| **CrypTen** | MPC | Meta | DenseNet-121, ResNet-50, BERT |
| **SecretFlow/SPU** | MPC | Ant Group | LLaMA-7B inference |
| **CrypTFlow2/EzPC** | MPC (2PC) | Microsoft Research | Chest X-ray (DenseNet) across 7 sites |
| **Concrete ML** | FHE | Zama | Quantised small-medium models |

### Combining HE and MPC with FL

In an FL pipeline, secure inference happens **after** federated training:

```
1. PSA   → align patient records across hospitals
2. FL    → train model across sites (with DP + SecAgg)
3. HE/MPC → run inference on new patients without revealing data
```

## References

- [TenSEAL](https://github.com/OpenMined/TenSEAL) — OpenMined, Apache-2.0
- [Microsoft SEAL](https://github.com/microsoft/SEAL) — MIT License
- [CrypTen](https://github.com/facebookresearch/CrypTen) — Meta, MIT License
- Cheon et al. (2017), *Homomorphic Encryption for Arithmetic of Approximate Numbers* (CKKS)
- Knott et al. (2021), *CrypTen: Secure Multi-Party Computation Meets Machine Learning*, NeurIPS

## Next Steps

- [PSA: Entity Alignment](psa-entity-alignment.ipynb) — pre-training stage
- [DP: Gradient Privacy](dp-gradient-privacy.ipynb) — during training
- [PET Reference](../reference/PET_Reference.md) — full technical comparison